# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mr-PeterMaged/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane 2 (Refresh / Content Opportunity Scoring) is a binary classification task whose output
feeds a ranking/scoring decision.**

Using the task-type table from the framing skill: "Will this one decline?" maps to
**classification** (a yes/no label from an observed outcome), while "Which ones first?" maps to
**ranking/scoring** (a priority score, evaluated with precision@K). Lane 2 needs both, in sequence:

1. **Classify** each page's probability of being a page that needs review (starting point: probability
   of `is_declining_label`, i.e. `trend_direction == "down"`).
2. **Rank** pages by that probability (optionally blended with a transparent rule score) so a
   capacity-limited reviewer sees the highest-priority pages first.

This is exactly the shape the already-run starter pipeline follows: `03_train_model.py` trains a
classifier, `04_evaluate_and_export.py` turns its probability into `final_refresh_score` and a ranked
queue. I'm keeping that shape for my capstone, but — as flagged in w01 — moving the *target* the
classifier predicts from a same-window proxy to a future-window observed outcome once I reach the
warehouse data contract (ML-04/05).

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

**Working proxy target (this notebook, starter data): `is_declining_label = (trend_direction == "down")`.**

I'm calling this a **proxy**, not an observed future outcome, on purpose — `trend_direction` is itself
derived from `trend_pct`, which is computed from the *same* trailing ~90-day window as the rest of the
features. It tells you "this page's recent window looks like a decline," not "this page declined after
the point where a reviewer would have acted." That's the label trap the data skill warns about:
`trend_direction` and `trend_pct` can be inspected as the proxy to beat, but must never be used as a
*feature* once a real model is trained on this or any stronger label.

For the capstone, the plan is to replace this proxy with a genuine future-window label built from the
warehouse's daily fact table — e.g. *features from the prior 90 days → decline (or recovery) observed
over the next 30 days* — which is measured strictly after the decision point. That data-contract work is
scheduled for ML-04/05, not this notebook; here I'm just sketching what the target column looks like on
the data I already have.

In [2]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Sketch the proxy target column exactly as the starter pipeline defines it.
# trend_direction / trend_pct are inspected here only to BUILD the proxy label -
# they are never used as model features (the label trap).
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df["is_declining_label"].value_counts())
print(f"\nproxy positive rate: {df['is_declining_label'].mean():.1%}")
df[["content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label"]].head()

is_declining_label
1    16262
0    13738
Name: count, dtype: int64

proxy positive rate: 54.2%


,content_id,client_id,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,down,-34.7,1


## 3. Success metric

**Primary: precision@50** — of the top 50 pages the ranking puts in front of a reviewer, how many
actually carry the proxy/target label? This is the metric that matches the real decision (a reviewer
has a fixed, small review budget), not overall accuracy across the full inventory, which would reward
getting the easy majority of low-priority pages right instead.

**Secondary/diagnostic: ROC-AUC and average precision** — used to judge the classifier's general
discrimination ability and to sanity-check precision@50 isn't a fluke of one particular K, before the
real review capacity (which may not be exactly 50) is nailed down later.

Naming this now, before any new training happens, matters because "good" defined after the fact always
looks good. I can already compute this metric today, on the already-run starter baseline, and I already
have a number to beat:

- rule baseline: **precision@50 = 0.240** (12 of the top 50 correct)
- starter random forest: **precision@50 = 0.740** (37 of the top 50 correct)

(from `outputs/model_report.md`, client-holdout validation — verified again below.)

In [3]:
# outputs/model_results.json is regenerated only when the pipeline is re-run and isn't committed;
# outputs/model_report.md IS committed and holds the verified numbers this notebook cites.
with open("../../outputs/model_report.md") as f:
    report = f.read()

for line in report.splitlines():
    if "baseline_rules" in line or "random_forest" in line:
        print(line.strip())

Best model: `random_forest` selected by `precision_at_50`.
| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |


## 4. The unit of analysis, as a real dataframe

**One row = one content page (`content_id`), scoped to one client (`client_id`), described by its
trailing ~90-day window of search and engagement signals.** Below is that slice as an actual dataframe —
the columns a reviewer's decision would realistically be based on, plus the proxy label from section 2.

In [4]:
unit_cols = [
    "content_id", "client_id", "content_type",
    "impressions_90d", "clicks_90d", "sessions_90d",
    "avg_position", "ctr", "engagement_rate", "scroll_rate",
    "content_age_days", "days_since_last_update",
    "is_declining_label",
]

unit_df = df[unit_cols]
print(f"shape: {unit_df.shape}  (one row per content page)")
print(f"content_id is unique per row: {unit_df['content_id'].is_unique}")
unit_df.head()

shape: (30000, 13)  (one row per content page)
content_id is unique per row: True


,content_id,client_id,content_type,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,scroll_rate,content_age_days,days_since_last_update,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,17,10.6,0.76,5.88,4.55,187,20,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,9,20.3,0.05,0.00,10.00,445,25,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,11,36.5,0.09,0.00,28.57,141,20,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,78,6.2,0.49,1.28,3.45,463,22,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,145,44.0,0.13,0.00,24.29,263,14,1


## 5. Why ML beats a fixed rule here

Two pieces of evidence, both already computable on this data:

1. **No single signal separates the label well.** The strongest single-feature correlation with the
   proxy label (computed below) is weak — meaning no one column crosses a clean threshold that would
   make a one-line if-statement work. The real pattern is several signals moving together (position,
   CTR, freshness, engagement, age), not one dominant variable.
2. **A multi-signal rule baseline already underperforms a learned model on the same data.** The
   starter's hand-written rule baseline — several if-statements combined — reaches precision@50 = 0.240,
   while a random forest trained on the same signals reaches 0.740 (section 3). If a hand-tuned
   *combination* of rules already leaves that much on the table, a single if-statement would do worse
   still.

Together, that's the ML/analysis case: the signal is real (a learned model clearly beats chance and
beats the rule baseline), but it's distributed across many correlated, moderate-strength signals rather
than concentrated in one variable a person could hand-code.

In [5]:
numeric_signals = [
    "avg_position", "ctr", "engagement_rate", "scroll_rate",
    "days_since_last_update", "content_age_days", "impressions_90d", "word_count",
]

corr_with_label = df[numeric_signals].corrwith(df["is_declining_label"]).sort_values(
    key=lambda s: s.abs(), ascending=False
)
print(corr_with_label.round(3))
print(f"\nstrongest single-feature correlation with the proxy label: {corr_with_label.abs().max():.3f}")
print("-> weak on its own; no single column is a usable threshold rule.")

content_age_days         -0.164
word_count                0.090
days_since_last_update    0.081
ctr                      -0.062
avg_position             -0.029
impressions_90d          -0.018
engagement_rate          -0.013
scroll_rate              -0.003
dtype: float64

strongest single-feature correlation with the proxy label: 0.164
-> weak on its own; no single column is a usable threshold rule.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.